# Capítulo 24 — MapReduce

**Livro:** começa na página **408** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import math, random, re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
random.seed(0)
print("Pronto. Aqui é tudo do zero: só Python puro (e matplotlib para gráficos).")

## Exemplo: Contagem de Palavras · PDF p. 409

#### 🧪 Exercício: MapReduce do zero

Escreva `map_reduce(entradas, mapper, reducer)` genérica (mapper gera pares chave-valor; agrupe por chave; reducer resume cada grupo) e use para contar palavras em 4 frases.

In [ ]:
def map_reduce(entradas, mapper, reducer):
    grupos = defaultdict(list)
    for e in entradas:
        for k, v in mapper(e): grupos[k].append(v)
    return [saida for k, vs in grupos.items() for saida in reducer(k, vs)]
def mapper(doc):
    for p in doc.lower().split(): yield p, 1
def reducer(palavra, contagens):
    yield palavra, sum(contagens)
docs = ["dados são legais", "dados grandes", "dados e mais dados", "python é legal"]
print(sorted(map_reduce(docs, mapper, reducer), key=lambda kv: -kv[1])[:3])

## Exemplo: Multiplicação de Matriz · PDF p. 415

#### 🧪 Exercício: Multiplicação de matrizes com MapReduce

Represente A (2×3) e B (3×2) como listas esparsas `(matriz, linha, coluna, valor)` e calcule C = A·B com MapReduce, conferindo com a multiplicação comum.

In [ ]:
A = [[1, 2, 0], [0, 3, 4]]; B = [[5, 0], [1, 2], [0, 6]]
esparsa = [("A", i, j, v) for i, l in enumerate(A) for j, v in enumerate(l) if v] + \
          [("B", i, j, v) for i, l in enumerate(B) for j, v in enumerate(l) if v]
linhas_A, colunas_B = 2, 2
def mapper(e):
    m, i, j, v = e
    if m == "A":
        for col in range(colunas_B): yield (i, col), (m, j, v)
    else:
        for lin in range(linhas_A): yield (lin, j), (m, i, v)
def reducer(chave, vals):
    a = {k: v for m, k, v in vals if m == "A"}; b = {k: v for m, k, v in vals if m == "B"}
    yield chave, sum(a[k] * b[k] for k in a if k in b)
grupos = defaultdict(list)
for e in esparsa:
    for k, v in mapper(e): grupos[k].append(v)
C = dict(kv for k, vs in grupos.items() for kv in reducer(k, vs))
print([[C[(i, j)] for j in range(2)] for i in range(2)])
print([[sum(A[i][k] * B[k][j] for k in range(3)) for j in range(2)] for i in range(2)])